# Landsat Satellite Classification

6,435 pixels from satellite images, each described by 36 multi-spectral values from a 3x3 neighbourhood. Predict the land-cover class of the central pixel — red soil, cotton crop, grey soil, damp grey soil, soil with vegetation stubble, or very damp grey soil.

In [1]:
import io, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def download(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=180) as r:
        return r.read()
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# UCI Statlog (Landsat Satellite) (id 146). Note: class 6 is absent from
# this dataset, so only 6 of the 7 numbered classes appear.
df = pd.read_csv(io.BytesIO(download('https://archive.ics.uci.edu/static/public/146/data.csv')))
print(df.shape)
print('Pixels per class code (class 6 absent):')
print(df['class'].value_counts().sort_index())
df.head(3)


(6435, 37)
Pixels per class code (class 6 absent):
class
1    1533
2     703
3    1358
4     626
5     707
7    1508
Name: count, dtype: int64


,Attribute1,Attribute2,Attribute3,Attribute4,Attribute5,Attribute6,Attribute7,Attribute8,Attribute9,Attribute10,...,Attribute28,Attribute29,Attribute30,Attribute31,Attribute32,Attribute33,Attribute34,Attribute35,Attribute36,class
0,92,115,120,94,84,102,106,79,84,102,...,104,88,121,128,100,84,107,113,87,3
1,84,102,106,79,84,102,102,83,80,102,...,100,84,107,113,87,84,99,104,79,3
2,84,102,102,83,80,102,102,79,84,94,...,87,84,99,104,79,84,99,104,79,3


In [2]:

feats = [c for c in df.columns if c != 'class']
X = df[feats]; y = df['class']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(Xtr, ytr)
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
acc_rf = accuracy_score(yte, rf.predict(Xte)); acc_lr = accuracy_score(yte, lr.predict(Xte))
print(f'Random Forest accuracy: {acc_rf:.4f}')
print(f'Logistic Regression accuracy: {acc_lr:.4f}')
print(classification_report(yte, rf.predict(Xte)))


Random Forest accuracy: 0.9093
Logistic Regression accuracy: 0.8552
              precision    recall  f1-score   support

           1       0.97      0.99      0.98       383
           2       0.98      0.97      0.97       176
           3       0.88      0.96      0.92       340
           4       0.74      0.56      0.64       156
           5       0.93      0.88      0.90       177
           7       0.89      0.91      0.90       377

    accuracy                           0.91      1609
   macro avg       0.90      0.88      0.89      1609
weighted avg       0.91      0.91      0.91      1609



In [3]:
imp = pd.Series(rf.feature_importances_, index=feats).sort_values(ascending=False)
print(imp.head(20))
imp.head(20).plot(kind='barh', title='Top 20 spectral features')
plt.tight_layout()
plt.show()
winner = rf if acc_rf >= acc_lr else lr
winner_name = 'Random Forest' if acc_rf >= acc_lr else 'Logistic Regression'
print(f'Winner: {winner_name}')
cm = confusion_matrix(yte, winner.predict(Xte), labels=sorted(y.unique()))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=sorted(y.unique()))
disp.plot(xticks_rotation=45)
plt.title(f'Confusion matrix - {winner_name}')
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.show()


Attribute17    0.072067
Attribute18    0.064824
Attribute20    0.054343
Attribute21    0.049883
Attribute22    0.045125
Attribute13    0.041829
Attribute16    0.037528
Attribute14    0.037327
Attribute24    0.035759
Attribute29    0.035479
Attribute6     0.030314
Attribute33    0.030189
Attribute9     0.029833
Attribute2     0.027867
Attribute28    0.027048
Attribute5     0.026957
Attribute25    0.026909
Attribute34    0.026807
Attribute30    0.024648
Attribute32    0.022842
dtype: float64


Winner: Random Forest
